# Les deux contrôles : les zones, puis les dates · *The two checks: zones, then dates*

Notebook compagnon de l'enquête **L'or monte-t-il parce que les monnaies s'effondrent ?** — [lire l'article](https://nmlab.io/ressources/prix-de-l-or-et-effondrement-des-monnaies).
Companion notebook to the study **Is gold rising because currencies are collapsing?**.

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure est régénérée par le code — un **schéma éditable** : changez les libellés à votre guise. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure from code — an **editable diagram**: change the labels as you like; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


# Contrôles publiés dans l'article : part de la hausse de l'indice-or retirée par
# la monnaie large. À gauche, le panier privé d'une zone ; à droite, quatre fenêtres.
# Published robustness checks: share of the gold index rise removed by broad money.
import numpy as np

DROP_ONE = [("JP", 66.0), ("UK", 61.0), ("CH", 61.0), ("EA", 60.7),
            ("US", 59.3), ("CA", 58.5), ("AU", 57.0), ("CN", 49.2)]
WINDOWS = [(1999, 2024, 79.5), (1999, 2025, 69.1), (1999, 2026, 59.0), (2015, 2026, 39.3)]
REFERENCE = 59.0                       # fenêtre principale, panier complet


from matplotlib.figure import Figure
from matplotlib.ticker import FuncFormatter

LABELS = {
    "fr": dict(
        title="Robuste à la composition du panier, sensible aux dates",
        sub="Part de la hausse de l'indice-or retirée par la monnaie large — les deux contrôles, à la même échelle.",
        left="Une zone retirée du panier", right="Une autre fenêtre",
        window="{a} T1 – {b} T1", reference="résultat publié : 59 %",
        zones=dict(US="États-Unis", EA="Zone euro", JP="Japon", UK="Royaume-Uni",
                   CH="Suisse", CA="Canada", AU="Australie", CN="Chine"),
        note="Aucune exclusion ne ramène la part près de zéro ; changer la date de fin la fait presque doubler.\n"
             "Le chiffre décrit donc la fenêtre annoncée : ce n'est ni un coefficient permanent, ni une part causale."),
    "en": dict(
        title="Robust to the basket's composition, sensitive to the dates",
        sub="Share of the gold index rise removed by broad money — both checks, on the same scale.",
        left="One zone dropped from the basket", right="A different window",
        window="{a} Q1 – {b} Q1", reference="published result: 59%",
        zones=dict(US="United States", EA="Euro area", JP="Japan", UK="United Kingdom",
                   CH="Switzerland", CA="Canada", AU="Australia", CN="China"),
        note="No exclusion brings the share near zero; changing the end date almost doubles it. The number\n"
             "describes the stated window: it is neither a permanent coefficient nor a causal share."),
}


def build_figure(drop_one: list, windows: list, reference: float, lang: str) -> Figure:
    """Deux panneaux de barres horizontales partageant le même axe des abscisses."""
    text = LABELS[lang]
    percent = (lambda v: f"{v:.1f} %".replace(".", ",") if lang == "fr" else f"{v:.1f}%")
    limit = 88

    fig = nm.figure(height_px=1060)
    left_ax = fig.add_axes([0.118, 0.175, 0.352, 0.545])
    right_ax = fig.add_axes([0.612, 0.175, 0.352, 0.545])

    left_labels = [text["zones"][zone] for zone, _ in drop_one][::-1]
    left_values = [share for _, share in drop_one][::-1]
    right_labels = [text["window"].format(a=a, b=b) for a, b, _ in windows][::-1]
    right_values = [share for _, _, share in windows][::-1]

    for ax, title, labels, values in ((left_ax, "left", left_labels, left_values),
                                      (right_ax, "right", right_labels, right_values)):
        positions = np.arange(len(values))
        colors = [nm.COLORS["text"] if abs(value - reference) < 0.05 else nm.COLORS["amber"]
                  for value in values]
        ax.barh(positions, values, height=0.62 if len(values) > 5 else 0.44, color=colors, zorder=3)
        for y, value, color in zip(positions, values, colors):
            ax.text(value + 2.2, y, percent(value), va="center", ha="left", fontsize=18.5,
                    color=color)
        ax.axvline(reference, color=nm.COLORS["text"], lw=2, ls=(0, (5, 4)), alpha=0.7, zorder=4)
        ax.set_yticks(positions, labels, fontsize=20)
        ax.set_xlim(0, limit)
        ax.set_ylim(-0.68, len(values) - 0.32)
        ax.xaxis.set_major_formatter(FuncFormatter(
            lambda v, _: f"{v:.0f} %".replace(".", ",") if lang == "fr" else f"{v:.0f}%"))
        ax.grid(axis="y", visible=False)
        ax.text(0.0, 1.055, text[title], transform=ax.transAxes, fontsize=23,
                fontweight="bold", color=nm.COLORS["text"], va="bottom")

    left_ax.text(reference + 2.4, -0.64, text["reference"], ha="left", va="bottom",
                 fontsize=17, color=nm.COLORS["muted"])

    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig


build_figure(DROP_ONE, WINDOWS, REFERENCE, LANG)